# ResNet50 — Feature Extraction & Invariance Analysis

Extracts features and computes invariance at **3 granularities**:

| Granularity | # levels | Description |
|-------------|-----------|-------------|
| **Per-block** | 5 | stem + output of each of the 4 macro-layers (layer1–4) |
| **Per-subblock** | 17 | stem + output of each of the 16 Bottleneck residual blocks |
| **Per-conv** | 49 | stem + feature after every conv inside every Bottleneck |

Inside each Bottleneck the three conv features are:
- **C1** — after `relu(bn1(conv1(x)))` = input to conv2  
- **C2** — after `relu(bn2(conv2(x)))` = input to conv3  
- **C3** — after `relu(bn3(conv3(x)) + shortcut)` = Bottleneck output (≡ sub-block feature)

Features are global-average-pooled to `(32, C)` vectors before computing invariance.
Invariance is summarised per layer as **(mean ± std)** over the 28 category pairs.

In [1]:
import os, sys
import torch
import numpy as np
from PIL import Image
import torchvision.models as models
import torchvision.transforms as T
from collections import OrderedDict
import matplotlib.pyplot as plt

sys.path.append('../utils')
import util

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


## Load ResNet50

In [2]:
weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights).to(device)
model.eval()

layer_objs = [model.layer1, model.layer2, model.layer3, model.layer4]
layer_sizes = [len(l) for l in layer_objs]   # [3, 4, 6, 3]
out_channels = [256, 512, 1024, 2048]

print('ResNet50 macro-layer structure:')
print('  stem : conv1(7x7,s=2) -> bn1 -> relu -> maxpool  ->  (B, 64, H/4, W/4)')
for li, (nsb, oc) in enumerate(zip(layer_sizes, out_channels)):
    print(f'  layer{li+1}: {nsb} Bottleneck sub-blocks  ->  (B, {oc}, ...)')
n_sb = sum(layer_sizes)
print(f'\nTotal sub-blocks : {n_sb}')
print(f'Per-block levels : 1 stem + 4 layers = 5')
print(f'Per-subblock     : 1 stem + {n_sb} = {n_sb + 1}')
print(f'Per-conv         : 1 stem + {n_sb} x 3 = {n_sb * 3 + 1}')

ResNet50 macro-layer structure:
  stem : conv1(7x7,s=2) -> bn1 -> relu -> maxpool  ->  (B, 64, H/4, W/4)
  layer1: 3 Bottleneck sub-blocks  ->  (B, 256, ...)
  layer2: 4 Bottleneck sub-blocks  ->  (B, 512, ...)
  layer3: 6 Bottleneck sub-blocks  ->  (B, 1024, ...)
  layer4: 3 Bottleneck sub-blocks  ->  (B, 2048, ...)

Total sub-blocks : 16
Per-block levels : 1 stem + 4 layers = 5
Per-subblock     : 1 stem + 16 = 17
Per-conv         : 1 stem + 16 x 3 = 49


## Load & preprocess images

In [3]:
from scipy.io import loadmat

datapath = '../data'
mat = loadmat(os.path.join(datapath, 'miguel_passive8x4.mat'))
img = mat['img'].astype(np.float32)
images_gray = np.transpose(img, (2, 0, 1))   # (32, 150, 600)
print('images_gray:', images_gray.shape, '  range:', images_gray.min(), images_gray.max())

nimg = len(images_gray)
img_class = np.repeat(np.arange(nimg // 4), 4)
print(f'nimg={nimg}  (8 categories x 4 instances)')

images_rgb = [Image.fromarray(images_gray[i]).convert('RGB') for i in range(nimg)]

images_gray: (32, 150, 600)   range: 0.0 255.0
nimg=32  (8 categories x 4 instances)


In [4]:
mean_norm = (0.485, 0.456, 0.406)
std_norm  = (0.229, 0.224, 0.225)

transform = T.Compose([
    T.Resize((64, 264)),
    T.ToTensor(),
    T.Normalize(mean=mean_norm, std=std_norm),
])

x = torch.stack([transform(im) for im in images_rgb], dim=0).to(device)
print('Input batch:', x.shape, x.dtype)

Input batch: torch.Size([32, 3, 64, 264]) torch.float32


## Register hooks and extract features

**Strategy:**
- `model.maxpool` forward hook → **stem** feature  
- `block.conv2` *pre*-hook → **C1** (input to conv2 = post-relu of conv1 branch)  
- `block.conv3` *pre*-hook → **C2** (input to conv3 = post-relu of conv2 branch)  
- `block` forward hook → **C3 / subblock** output (post-relu residual add)

Using pre-hooks on conv2 / conv3 safely captures post-relu activations even though  
`self.relu` is shared and applied in-place inside the Bottleneck forward.

In [5]:
feats_per_conv     = OrderedDict()
feats_per_subblock = OrderedDict()
handles = []

def make_hook(store, name):
    """Forward hook: saves the module output."""
    def _h(m, inp, out):
        store[name] = out.detach().cpu().clone()
    return _h

def make_pre_hook(store, name):
    """Pre-hook: saves inp[0] = post-activation output of the previous operation."""
    def _h(m, inp):
        store[name] = inp[0].detach().cpu().clone()
    return _h

# --- Stem ---
handles.append(model.maxpool.register_forward_hook(make_hook(feats_per_conv,     'stem')))
handles.append(model.maxpool.register_forward_hook(make_hook(feats_per_subblock, 'stem')))

# --- Bottleneck layers ---
for li, layer in enumerate(layer_objs):
    for bi, block in enumerate(layer):
        sb = f'L{li+1}.B{bi}'
        # C1: relu(bn1(conv1(x)))  — captured as input to conv2
        handles.append(block.conv2.register_forward_pre_hook(make_pre_hook(feats_per_conv, f'{sb}.C1')))
        # C2: relu(bn2(conv2(x)))  — captured as input to conv3
        handles.append(block.conv3.register_forward_pre_hook(make_pre_hook(feats_per_conv, f'{sb}.C2')))
        # C3 / subblock output: relu(bn3(conv3(x)) + shortcut)
        handles.append(block.register_forward_hook(make_hook(feats_per_conv,     f'{sb}.C3')))
        handles.append(block.register_forward_hook(make_hook(feats_per_subblock, sb)))

with torch.no_grad():
    _ = model(x)

for h in handles:
    h.remove()

# Per-block: last subblock of each macro-layer = that layer's output
feats_per_block = OrderedDict()
feats_per_block['stem'] = feats_per_subblock['stem']
for li, layer in enumerate(layer_objs):
    feats_per_block[f'L{li+1}'] = feats_per_subblock[f'L{li+1}.B{len(layer)-1}']

print(f'Per-conv     : {len(feats_per_conv)} features')
print(f'Per-subblock : {len(feats_per_subblock)} features')
print(f'Per-block    : {len(feats_per_block)} features')
print('\nExample shapes:')
for k in ['stem', 'L1.B0.C1', 'L1.B0.C2', 'L1.B0.C3', 'L2.B0.C1', 'L4.B2.C3']:
    print(f'  {k:16s}: {tuple(feats_per_conv[k].shape)}')

Per-conv     : 49 features
Per-subblock : 17 features
Per-block    : 5 features

Example shapes:
  stem            : (32, 64, 16, 66)
  L1.B0.C1        : (32, 64, 16, 66)
  L1.B0.C2        : (32, 64, 16, 66)
  L1.B0.C3        : (32, 256, 16, 66)
  L2.B0.C1        : (32, 128, 16, 66)
  L4.B2.C3        : (32, 2048, 2, 9)


## Global average pool → (B, C) feature vectors

In [6]:
def pool_features(feats_dict):
    """(B, C, H, W) -> (B, C) via spatial average pooling."""
    pooled = OrderedDict()
    for name, t in feats_dict.items():
        pooled[name] = t.mean(dim=[2, 3]).numpy() if t.ndim == 4 else t.numpy()
    return pooled

pooled_conv     = pool_features(feats_per_conv)
pooled_subblock = pool_features(feats_per_subblock)
pooled_block    = pool_features(feats_per_block)

print('Per-block pooled shapes:')
for k, v in pooled_block.items():
    print(f'  {k}: {v.shape}')

Per-block pooled shapes:
  stem: (32, 64)
  L1: (32, 256)
  L2: (32, 512)
  L3: (32, 1024)
  L4: (32, 2048)


## Compute invariance

For each granularity, build a representation similarity matrix (Pearson correlation) across all images per layer, then compute the pair-wise invariance index as:

$$\text{inv}(A,B) = \frac{\rho_{AA} + \rho_{BB}}{2} - \rho_{AB}$$

where $\rho_{XY}$ is the mean within/across-category similarity.

In [7]:
def compute_invariance(pooled_dict):
    """Compute pair-invariance DataFrame for all layers in pooled_dict."""
    keys = list(pooled_dict.keys())
    arr = np.empty(len(keys), dtype=object)
    for i, k in enumerate(keys):
        arr[i] = pooled_dict[k]   # (32, C)
    rep_mtx  = util.compute_model_rep_mtx(arr)
    invar_df = util.compute_pair_inv_model(rep_mtx)
    return invar_df

print('Computing per-block invariance   ...')
invar_df_block    = compute_invariance(pooled_block)
print('Computing per-subblock invariance...')
invar_df_subblock = compute_invariance(pooled_subblock)
print('Computing per-conv invariance    ...')
invar_df_conv     = compute_invariance(pooled_conv)
print('Done.')

Computing per-block invariance   ...
Computing per-subblock invariance...
Computing per-conv invariance    ...
Done.


## Summarise (mean ± std per layer) and save

In [8]:
def summarise_inv(invar_df, layer_names):
    """Returns dict with layer_names, mean, std arrays."""
    grp = invar_df.groupby('layer')['pair_invariance']
    return {
        'layer_names': layer_names,
        'mean': grp.mean().values,
        'std':  grp.std().values,
    }

keys_block    = list(pooled_block.keys())
keys_subblock = list(pooled_subblock.keys())
keys_conv     = list(pooled_conv.keys())

inv_block    = summarise_inv(invar_df_block,    keys_block)
inv_subblock = summarise_inv(invar_df_subblock, keys_subblock)
inv_conv     = summarise_inv(invar_df_conv,     keys_conv)

os.makedirs('outputs', exist_ok=True)
np.save('outputs/resnet50_inv_per_block.npy',    inv_block,    allow_pickle=True)
np.save('outputs/resnet50_inv_per_subblock.npy', inv_subblock, allow_pickle=True)
np.save('outputs/resnet50_inv_per_conv.npy',     inv_conv,     allow_pickle=True)
print('Invariance summaries saved.')
print('\nPer-block mean invariance:')
for name, m, s in zip(keys_block, inv_block['mean'], inv_block['std']):
    print(f'  {name:6s}: {m:.4f} ± {s:.4f}')

Invariance summaries saved.

Per-block mean invariance:
  stem  : 0.0016 ± 0.0019
  L1    : 0.0346 ± 0.0344
  L2    : 0.1299 ± 0.0814
  L3    : 0.2730 ± 0.0751
  L4    : 0.5259 ± 0.0958


## Save pooled features
Block-level file overwrites the one produced by `models_ResNet50.ipynb` (same 5 layers).

In [9]:
def save_pooled_arr(pooled_dict, fname):
    arr = np.empty(len(pooled_dict), dtype=object)
    for i, v in enumerate(pooled_dict.values()):
        arr[i] = v
    np.save(fname, arr, allow_pickle=True)
    print('saved:', fname)

save_pooled_arr(pooled_block,    'outputs/resnet50_pooled_features.npy')          # compatible with invariance_cnn.ipynb
save_pooled_arr(pooled_subblock, 'outputs/resnet50_pooled_features_subblock.npy')
save_pooled_arr(pooled_conv,     'outputs/resnet50_pooled_features_conv.npy')

saved: outputs/resnet50_pooled_features.npy
saved: outputs/resnet50_pooled_features_subblock.npy
saved: outputs/resnet50_pooled_features_conv.npy


## Plot invariance across all 3 granularities

In [ ]:
def plot_inv_summary(ax, inv, title, color='#c0394b', label_rotation=45, label_fs=9):
    names = inv['layer_names']
    mean  = inv['mean']
    std   = inv['std']
    x     = np.arange(len(names))
    ax.plot(x, mean, 'o-', color=color, lw=2, ms=5)
    ax.fill_between(x, mean - std, mean + std, alpha=0.2, color=color)
    ax.axhline(0, color='gray', lw=1, linestyle='--')
    ax.set_ylim(-0.1, 0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=label_rotation, ha='right', fontsize=label_fs)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel('Invariance')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


def add_block_boundaries(ax, n_per_layer):
    """Draw vertical dividers and layer labels where each macro-layer starts.
    n_per_layer: number of entries per macro-layer (e.g. 3,4,6,3 for subblock,
    or 9,12,18,9 for per-conv).
    """
    offset = 1   # index 0 is stem
    for lname, nsb in zip(['L1', 'L2', 'L3', 'L4'], n_per_layer):
        ax.axvline(offset - 0.5, color='gray', lw=0.8, linestyle=':', alpha=0.7)
        # use axes-fraction y so the label is always near the top
        ax.text(offset + 0.1, 0.97, lname,
                transform=ax.get_xaxis_transform(),
                ha='left', va='top', fontsize=8, color='gray')
        offset += nsb


fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Panel 1: per-block (5 points, easy to read)
plot_inv_summary(axes[0], inv_block,
                 f'Per-Block  ({len(keys_block)} levels)',
                 color='#c0394b', label_rotation=30, label_fs=10)

# Panel 2: per-subblock (17 points)
plot_inv_summary(axes[1], inv_subblock,
                 f'Per-Sub-Block  ({len(keys_subblock)} levels)',
                 color='#c0394b', label_rotation=90, label_fs=6)
add_block_boundaries(axes[1], layer_sizes)            # [3, 4, 6, 3]

# Panel 3: per-conv (49 points)
plot_inv_summary(axes[2], inv_conv,
                 f'Per-Conv  ({len(keys_conv)} levels)',
                 color='#c0394b', label_rotation=90, label_fs=4)
add_block_boundaries(axes[2], [n * 3 for n in layer_sizes])   # [9, 12, 18, 9]

plt.suptitle('ResNet50 — Invariance Across Layers', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/resnet50_invariance.png', dpi=300, bbox_inches='tight')
plt.show()